## Interactive Dashboard for Energy Distribution
Modern energy distribution systems depend heavily on digital technologies to communicate complex information to both utility companies and end users. Visualizations play a central role in this process, enabling stakeholders to interpret consumption patterns, identify inefficiencies, and support evidence based decision making. To address these needs, an interactive dashboard was developed using Python, Dash, and Plotly. 
The dashboard is designed for two primary user groups: 
-	Utility companies, who require clear insights into regional energy consumption trends, anomalies, and long term patterns.
-	Customers, who benefit from understanding per capita consumption and comparing performance across European countries.
The dashboard presents both total energy consumption and per capita metrics, enabling users to explore data by year and country(clean_energy.csv). Interactive elements such as dropdown filters and hover activated trend charts support dynamic exploration.
The visualizations minimize non essential elements. Dark backgrounds reduce visual noise, while colour bar and axes are compact and clearly labelled. Every graphical component contributes directly to communicating data.
The dashboard avoids unnecessary decoration such as 3D effects, gradients, or distracting icons. Clean lines, simple colour palettes, and intuitive scales ensure that the focus remains on the underlying data.
The choropleth map enables cross country comparison for a selected year, while the line chart provides temporal comparison for individual countries. This dual view structure supports both spatial and longitudinal analysis. Although not displayed as separate static panels, the dashboard effectively implements the concept of small multiples through interactive filtering. Users can switch between metrics and years, generating multiple comparative views on demand. Textual elements such as titles, labels, and hover tooltips are integrated directly into the visualizations. This reduces reliance on external legends and supports immediate comprehension.
The dashboard is fully interactive, allowing users to explore data at their own pace. Dropdown menus and hover events respond instantly, and good design should invite exploration and discovery.


In [2]:
import pandas as pd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

df = pd.read_csv("clean_energy.csv")
df.head()

colors = {
    'background': '#111111',
    'text': '#C5DB5F'
}

app = dash.Dash(__name__)

app.layout = html.Div(style={'backgroundColor': colors['background']}, children=[

    html.H1("Energy Analytics Dashboard",
            style={'text-align': 'center', 'color': colors['text']}),

    html.Div([

        html.Div([
            dcc.Dropdown(
                id="slct_year",
                options=[{"label": str(y), "value": y} for y in sorted(df['Year'].unique())],
                multi=False,
                value=df['Year'].max(),
                style={'width': "70%"}
            ),
            html.Div(id='output_container', style={'color': colors['text'], 'padding-top': '10px'}),
        ], style={'width': '48%', 'display': 'inline-block', 'vertical-align': 'top'}),

        html.Div([
            dcc.Dropdown(
                id="slct_metric",
                options=[
                    {"label": "Total Consumption (KTOE)", "value": "Consumption"},
                    {"label": "Consumption per Capita (TOE)", "value": "Consumption_per_capita_TOE"}
                ],
                multi=False,
                value="Consumption_per_capita_TOE",
                style={'width': "80%"}
            ),
        ], style={'width': '49%', 'text-align': 'center', 'display': 'inline-block', 'vertical-align': 'top'})

    ], style={'padding': '10px 5px'}),

    html.Div([

        html.Div(
            children=[
                dcc.Graph(id='energy_map')
            ],
            style={
                'flex': '1',
                'padding': '10px',
                'min-width': '0'
            }
        ),

        html.Div(
            children=[
                dcc.Graph(id='trend_chart')
            ],
            style={
                'flex': '1',
                'padding': '10px',
                'min-width': '0'
            }
        )

    ],
    style={
        'display': 'flex',
        'flex-direction': 'row',
        'justify-content': 'space-between',
        'width': '100%'
    })

])

@app.callback(
    [Output('output_container', 'children'),
     Output('energy_map', 'figure')],
    [Input('slct_year', 'value'),
     Input('slct_metric', 'value')]
)
def update_map(slct_year, slct_metric):

    container = f"The year chosen by user was: {slct_year}"

    dff = df[df['Year'] == slct_year]

    labels = {
        "Consumption": "Total Consumption (KTOE)",
        "Consumption_per_capita_TOE": "Consumption per Capita (TOE)"
    }

    fig = px.choropleth(
        dff,
        locations="Country",
        locationmode="country names",
        color=slct_metric,
        hover_data=["Country", "Consumption", "Population", "Consumption_per_capita_TOE"],
        custom_data=["Country"],
        color_continuous_scale=px.colors.sequential.Blugrn,
        labels={slct_metric: labels[slct_metric]},
        template='plotly_dark',
        scope="europe"
    )

    fig.update_layout(
        margin={"r": 0, "t": 10, "l": 0, "b": 0},
        coloraxis_colorbar=dict(
            thickness=12,
            len=0.5,
            y=0.5
        )
    )

    return container, fig

def create_chart(df_new, slct_metric, country_name):

    df_new = df_new[df_new['Country'] == country_name]

    labels = {
        "Consumption": "Total Consumption (KTOE)",
        "Consumption_per_capita_TOE": "Consumption per Capita (TOE)"
    }

    fig = px.line(
        df_new,
        x='Year',
        y=slct_metric,
        title=f"{country_name} - {labels[slct_metric]}",
        template='plotly_dark'
    )

    fig.update_layout(
        title={'xanchor': 'center', 'yanchor': 'top', 'y': 0.9, 'x': 0.5},
        margin={"r": 20, "t": 40, "l": 20, "b": 20}
    )

    return fig

@app.callback(
    Output('trend_chart', 'figure'),
    [Input('energy_map', 'hoverData'),
     Input('slct_metric', 'value')]
)
def update_chart(hoverData, slct_metric):

    df_new = df.copy()

    if hoverData is None:
        country_name = df_new['Country'].iloc[0]
    else:
        country_name = hoverData['points'][0]['customdata'][0]

    return create_chart(df_new, slct_metric, country_name)


if __name__ == '__main__':
    app.run(debug=True)